### 1.检查接口以及manifest

In [1]:
from pathlib import Path
import pandas as pd
import inspect
import sys

root = Path(r"D:\Projects\speech-ai-learning")
metadata_dir = root / "metadata"

if root not in sys.path:
    sys.path.insert(0, str(root))

print("===== Metadata CSV 文件 =====")

for path in sorted(metadata_dir.glob("*.csv")):
    print(path.name)

print("\n===== 现有 Manifest =====")

manifest_path = metadata_dir / "manifest.csv"

if manifest_path.exists():
    df = pd.read_csv(manifest_path)

    print("样本数:", len(df))
    print("字段:", df.columns.tolist())

    for col in ["split", "language", "locale"]:
        if col in df.columns:
            print(f"\n{col}:")
            print(df[col].value_counts(dropna=False).head(15))
else:
    print("manifest.csv 不存在")

print("\n===== 已有评估接口 =====")

from src import metrics, normalize, asr_inference

for module, names in [
    (metrics, ["score_tokens", "score_corpus"]),
    (normalize, ["normalize_score"]),
    (asr_inference, ["load_whisper_model", "transcribe_file"]),
]:
    for name in names:
        fn = getattr(module, name, None)

        print(
            module.__name__,
            name,
            inspect.signature(fn) if fn else "未找到"
        )

===== Metadata CSV 文件 =====
dev_candidates_W7D5.csv
manifest.csv
source_registry.csv
test_candidates.csv
test_candidates_pre_W6D4.csv
test_candidates_pre_W6D5.csv

===== 现有 Manifest =====
样本数: 2860
字段: ['audio_path', 'speaker_id', 'text', 'source_id', 'parent_uid', 'start_time', 'end_time', 'split']

split:
split
train    2281
dev       409
test      170
Name: count, dtype: int64

===== 已有评估接口 =====
src.metrics score_tokens (reference_tokens, hypothesis_tokens)
src.metrics score_corpus (token_pairs)
src.normalize normalize_score (text: str) -> str
src.asr_inference load_whisper_model (checkpoint='openai/whisper-small', device=None)
src.asr_inference transcribe_file (audio_path, processor, model, device=None, language='zh', task='transcribe', max_new_tokens=96)


### 2.检查数据集

In [2]:
#1.流式加载数据集
from datasets import load_dataset, Audio

english_dev = load_dataset(
    "openslr/librispeech_asr",
    "clean",
    split="validation",
    streaming=True,
)

# 关闭自动音频解码
english_dev = english_dev.cast_column(
    "audio",
    Audio(sampling_rate=16000, decode=False)
)

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

In [3]:
#直接读取前20条样本
samples = list(english_dev.take(20))

print("抽取数量:", len(samples))

for sample in samples[:3]:
    print("ID:", sample["id"])
    print("Speaker:", sample["speaker_id"])
    print("Reference:", sample["text"])
    print("Audio information:", sample["audio"].keys())
    print("-" * 50)

抽取数量: 20
ID: 2277-149896-0000
Speaker: 2277
Reference: HE WAS IN A FEVERED STATE OF MIND OWING TO THE BLIGHT HIS WIFE'S ACTION THREATENED TO CAST UPON HIS ENTIRE FUTURE
Audio information: dict_keys(['bytes', 'path'])
--------------------------------------------------
ID: 2277-149896-0001
Speaker: 2277
Reference: HE WOULD HAVE TO PAY HER THE MONEY WHICH SHE WOULD NOW REGULARLY DEMAND OR THERE WOULD BE TROUBLE IT DID NOT MATTER WHAT HE DID
Audio information: dict_keys(['bytes', 'path'])
--------------------------------------------------
ID: 2277-149896-0002
Speaker: 2277
Reference: HURSTWOOD WALKED THE FLOOR MENTALLY ARRANGING THE CHIEF POINTS OF HIS SITUATION
Audio information: dict_keys(['bytes', 'path'])
--------------------------------------------------


In [4]:
#检查原始音频数据
audio_info = samples[0]["audio"]

print("音频信息:", audio_info)
print("音频数据类型:", type(audio_info["bytes"]))

if audio_info["bytes"] is not None:
    print("音频字节大小:", len(audio_info["bytes"]))
else:
    print("音频文件路径:", audio_info["path"])

音频信息: {'bytes': b'fLaC\x00\x00\x00"\x10\x00\x10\x00\x00\tW\x00\x15\x1a\x03\xe8\x00\xf0\x00\x01\x9b\xe0\x15\x90\xba\x94\xff\xf5\x8c\xa5j"\x9b\xf1\xe9PZq\x03\x00\x00\x12\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x10\x00\x04\x00\x00( \x00\x00\x00reference libFLAC 1.2.1 20070917\x00\x00\x00\x00\x81\x00 \x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x

### 3.解码并保存20条音频

In [5]:
#测试第一条音频
import io
import soundfile as sf

audio_bytes = samples[0]["audio"]["bytes"]

waveform, sr = sf.read(
    io.BytesIO(audio_bytes),
    dtype="float32"
)

print("Waveform shape:", waveform.shape)
print("Sampling rate:", sr)
print("Duration:", len(waveform) / sr, "seconds")

Waveform shape: (105440,)
Sampling rate: 16000
Duration: 6.59 seconds


In [10]:
#批量保存20条音频
from pathlib import Path
import io
import soundfile as sf
import pandas as pd
import sys

#创建目录
root  = Path.cwd()
while not (root / "src").exists():
    if root == root.parent:
        raise FileNotFoundError("无法定位项目根目录")
    root = root.parent

audio_dir = root / "data" / "w08_english_dev_20"

#audio_dir = Path("data/w08_english_dev_20")
audio_dir.mkdir(parents=True,exist_ok=True)

metadata = []

for i, sample in enumerate(samples, start=1):

    #获取原始音频字节
    audio_bytes = sample["audio"]["bytes"]

    #FLAC解码
    waveform, sr = sf.read(
        io.BytesIO(audio_bytes),
        dtype="float32"
    )

    #保存为WAV
    filename = f"{i:03d}_{sample['id']}.wav"
    audio_path = audio_dir / filename

    sf.write(
        audio_path,
        waveform,
        sr,
        subtype="PCM_16"
    )

    #保存元数据
    metadata.append({
        "id": sample["id"],
        "speaker_id": sample["speaker_id"],
        "reference": sample["text"],
        "audio_path": str(audio_path.as_posix()),
        "sampling_rate": sr,
        "duration": len(waveform) / sr
    })

#保存CSV
metadata_dir = root / "metadata"
metadata_dir.mkdir(parents=True,exist_ok=True)

df = pd.DataFrame(metadata)

df.to_csv(
    metadata_dir / "w08_english_dev_20.csv",
    index=False,
    encoding="utf-8-sig"
)

print("保存完成！")
print("音频数量:", len(df))
print(df.head())

保存完成！
音频数量: 20
                 id  speaker_id  \
0  2277-149896-0000        2277   
1  2277-149896-0001        2277   
2  2277-149896-0002        2277   
3  2277-149896-0003        2277   
4  2277-149896-0004        2277   

                                           reference  \
0  HE WAS IN A FEVERED STATE OF MIND OWING TO THE...   
1  HE WOULD HAVE TO PAY HER THE MONEY WHICH SHE W...   
2  HURSTWOOD WALKED THE FLOOR MENTALLY ARRANGING ...   
3         HE ALSO THOUGHT OF HIS MANAGERIAL POSITION   
4                 HOW WOULD THE PAPERS TALK ABOUT IT   

                                          audio_path  sampling_rate  duration  
0  d:/Projects/speech-ai-learning/data/w08_englis...          16000     6.590  
1  d:/Projects/speech-ai-learning/data/w08_englis...          16000     7.145  
2  d:/Projects/speech-ai-learning/data/w08_englis...          16000     4.835  
3  d:/Projects/speech-ai-learning/data/w08_englis...          16000     2.840  
4  d:/Projects/speech-ai-learning/dat

In [7]:
#对数据重新检验speaker
from collections import Counter

# 统计说话人分布
speaker_counts = Counter(
    sample["speaker_id"] for sample in samples
)

# 统计章节分布
chapter_counts = Counter(
    sample["chapter_id"] for sample in samples
)

print("===== Speaker Distribution =====")
print(speaker_counts)

print("\n===== Chapter Distribution =====")
print(chapter_counts)

print("\n不同说话人数:", len(speaker_counts))
print("不同章节数:", len(chapter_counts))

===== Speaker Distribution =====
Counter({2277: 20})

===== Chapter Distribution =====
Counter({149896: 20})

不同说话人数: 1
不同章节数: 1


### 4.加载模型并完成单条推理

In [11]:
#1.检查环境和音频路径
import sys
import torch
import transformers
import soundfile as sf

root = Path(r"D:\Projects\speech-ai-learning")
if root not in sys.path:
    sys.path.insert(0, str(root))

from pathlib import Path
from src.audio_utils import (
    load_audio,
    to_mono,
    resample_audio,
)

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:",torch.cuda.get_device_name(0))

#定位项目根目录
PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "src").exists():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("无法定位项目根目录")
    PROJECT_ROOT = PROJECT_ROOT.parent

# 读取20条英文音频清单
import pandas as pd

metadata_path = (
    PROJECT_ROOT
    / "metadata"
    / "w08_english_dev_20.csv"
)

df = pd.read_csv(metadata_path)

audio_path = Path(df.loc[0, "audio_path"])

if not audio_path.is_absolute():
    audio_path = PROJECT_ROOT / audio_path

print("\nAudio path:", audio_path)
print("Audio exists:", audio_path.exists())

Python: d:\anaconda3\envs\speech-ai\python.exe
PyTorch: 2.14.1+cu132
Transformers: 5.21.0
CUDA available: True
GPU: NVIDIA GeForce RTX 5070 Laptop GPU

Audio path: d:\Projects\speech-ai-learning\data\w08_english_dev_20\001_2277-149896-0000.wav
Audio exists: True


In [ ]:
#加载wav2vec2模型
import torch

from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
)

CHECKPOINT = "facebook/wav2vec2-base-960h"

def load_wav2vec2_model(
    checkpoint=CHECKPOINT,
    device=None
):
    """
    加载 wav2vec2 CTC 模型和 Processor。

    返回：
        processor
        model
        device
    """

    #自动选择设备
    if device is None:
        device = (
            "cuda"
            if torch.cuda.is_available()
            else "cpu"
        )

    device = torch.device(device)

    #加载Processor
    processor = Wav2Vec2Processor.from_pretrained(
        checkpoint
    )

    # 加载 CTC 模型
    model = Wav2Vec2ForCTC.from_pretrained(
        checkpoint
    )

    model = model.to(device)

    #切换评估模式
    model.eval()

    print("Checkpoint:", checkpoint)
    print("Device:", device)
    print(
        "Sampling rate:",
        processor.feature_extractor.sampling_rate
    )
    print(
        "Vocabulary size:",
        model.config.vocab_size
    )

    return processor, model, device


processor, model, device = load_wav2vec2_model()

In [13]:
#编写单条音频推理函数
import numpy as np

def transcribe_wav2vec2_file(
    audio_path,
    processor,
    model,
    device=None,
):
    """
    对单条英文音频进行 wav2vec2 CTC 推理。

    返回：
        transcript
        predicted_ids
        metadata
    """

    audio_path = Path(audio_path)

    # 1. 确定设备
    if device is None:
        device = next(model.parameters()).device
    else:
        device = torch.device(device)

    model.eval()

    #2.读取音频
    waveform, original_sr = load_audio(audio_path)

    #waveform: (C,T)

    #3.转换为单声道
    waveform = to_mono(waveform)

    #4.获取模型要求的采样率
    target_sr = processor.feature_extractor.sampling_rate

    #必要时重采样
    waveform = resample_audio(
        waveform,
        orig_sr=original_sr,
        target_sr=target_sr
    )

    # 6. 转换为 NumPy 一维数组
    audio_np = (
        waveform.squeeze(0)
        .cpu()
        .numpy()
        .astype(np.float32, copy=False)
    )

    #7. Processor推理
    inputs = processor(
        audio_np,
        sampling_rate=target_sr,
        return_tensors="pt"
    )

    input_values = inputs.input_values.to(device)

    #8.模型推理
    with torch.inference_mode():
        outputs = model(
            input_values=input_values
        )

        logits = outputs.logits

    #9.Argmax
    predicted_ids = torch.argmax(
        logits,
        dim=-1
    )

    # 10. CTC Greedy Decoding
    transcript = processor.batch_decode(
        predicted_ids.detach().cpu().numpy()
    )[0]

    # 11. 记录元数据
    metadata = {
        "audio_path": str(audio_path),
        "checkpoint": CHECKPOINT,
        "device": str(device),
        "original_sampling_rate": original_sr,
        "model_sampling_rate": target_sr,
        "duration_sec": len(audio_np) / target_sr,
        "waveform_shape": tuple(waveform.shape),
        "input_values_shape": tuple(input_values.shape),
        "logits_shape": tuple(logits.shape),
        "predicted_ids_shape": tuple(predicted_ids.shape),
    }

    return {
        "transcript": transcript,
        "predicted_ids": predicted_ids.cpu().tolist()[0],
        "metadata": metadata,
    }

In [14]:
#执行第一条音频推理
result = transcribe_wav2vec2_file(
    audio_path=audio_path,
    processor=processor,
    model=model,
    device=device
)

print("\n========== Wav2Vec2 ASR Result ==========")

print("\nReference:")
print(df.loc[0, "reference"])

print("\nHypothesis:")
print(result["transcript"])

print("\nPredicted IDs:")
print(result["predicted_ids"][:30])

print("\nMetadata:")

for key, value in result["metadata"].items():
    print(f"{key}: {value}")

2026-11-13 02:32:37,190 - INFO - Loaded d:\Projects\speech-ai-learning\data\w08_english_dev_20\001_2277-149896-0000.wav,shape=torch.Size([1, 105440]), sr=16000



========== Wav2Vec2 ASR Result ==========

Reference:
HE WAS IN A FEVERED STATE OF MIND OWING TO THE BLIGHT HIS WIFE'S ACTION THREATENED TO CAST UPON HIS ENTIRE FUTURE

Hypothesis:
HE WAS IN A FEVERED STATE OF MIND OWING TO THE BLIGHT HIS WIFE'S ACTION THREATENED TO CAST UPON HIS ENTIRE FUTURE

Predicted IDs:
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 11, 11, 5, 0, 4, 18, 18, 0, 7, 12, 0, 4, 4, 0, 10]

Metadata:
audio_path: d:\Projects\speech-ai-learning\data\w08_english_dev_20\001_2277-149896-0000.wav
checkpoint: facebook/wav2vec2-base-960h
device: cuda
original_sampling_rate: 16000
model_sampling_rate: 16000
duration_sec: 6.59
waveform_shape: (1, 105440)
input_values_shape: (1, 105440)
logits_shape: (1, 329, 32)
predicted_ids_shape: (1, 329)


### 完成20条音频的批量推理

In [ ]:

print("Processor:", processor is not None)
print("Model:", model is not None)
print("Device:", device)

print("推理函数:", transcribe_wav2vec2_file)

Processor: True
Model: True
Device: cuda
推理函数: <function transcribe_wav2vec2_file at 0x00000284DE72CA40>


In [16]:
#读取固定的20条音频清单
PROJECT_ROOT = Path(r"D:\Projects\speech-ai-learning")

metadata_path = (
    PROJECT_ROOT
    / "metadata"
    / "w08_english_dev_20.csv"
)

batch_df = pd.read_csv(metadata_path)

print("样本数量:", len(batch_df))
print("字段:", batch_df.columns.tolist())

#基础检查
assert len(batch_df) == 20
assert batch_df["id"].is_unique
assert batch_df["reference"].notna().all()
assert batch_df["audio_path"].notna().all()

print("数据检查通过！")

样本数量: 20
字段: ['id', 'speaker_id', 'reference', 'audio_path', 'sampling_rate', 'duration']
数据检查通过！


In [17]:
#批量推理
from pathlib import Path

prediction_rows = []

for index, row in batch_df.iterrows():

    #1.获取音频路径
    audio_path = Path(row["audio_path"])

    if not audio_path.is_absolute():
        audio_path = PROJECT_ROOT / audio_path

    if not audio_path.exists():
        raise FileNotFoundError(audio_path)

    #2. 调用上一模块的单条推理函数
    result = transcribe_wav2vec2_file(
        audio_path=audio_path,
        processor=processor,
        model=model,
        device=device,
    )

    #3.获取预测文本和元数据
    hypothesis = result["transcript"]
    meta = result["metadata"]

    #4. 保存本条结果
    prediction_rows.append({
        "id": row["id"],
        "speaker_id": row["speaker_id"],
        "reference": row["reference"],
        "hypothesis": hypothesis,
        "audio_path": row["audio_path"],
        "checkpoint": meta["checkpoint"],
        "device": meta["device"],
        "sampling_rate": meta["model_sampling_rate"],
        "duration_sec": meta["duration_sec"],
        "input_length": meta["input_values_shape"][1],
        "logits_time_steps": meta["logits_shape"][1],
        "vocab_size": meta["logits_shape"][2],
    })

    # 5. 打印进度
    print(
        f"[{index + 1:02d}/20] "
        f"{row['id']} 推理完成"
    )

print("\n全部推理完成！")

2026-11-13 02:48:58,709 - INFO - Loaded d:\Projects\speech-ai-learning\data\w08_english_dev_20\001_2277-149896-0000.wav,shape=torch.Size([1, 105440]), sr=16000
2026-11-13 02:48:59,469 - INFO - Loaded d:\Projects\speech-ai-learning\data\w08_english_dev_20\002_2277-149896-0001.wav,shape=torch.Size([1, 114320]), sr=16000
2026-11-13 02:48:59,511 - INFO - Loaded d:\Projects\speech-ai-learning\data\w08_english_dev_20\003_2277-149896-0002.wav,shape=torch.Size([1, 77360]), sr=16000
2026-11-13 02:48:59,545 - INFO - Loaded d:\Projects\speech-ai-learning\data\w08_english_dev_20\004_2277-149896-0003.wav,shape=torch.Size([1, 45440]), sr=16000
2026-11-13 02:48:59,588 - INFO - Loaded d:\Projects\speech-ai-learning\data\w08_english_dev_20\005_2277-149896-0004.wav,shape=torch.Size([1, 31280]), sr=16000


[01/20] 2277-149896-0000 推理完成
[02/20] 2277-149896-0001 推理完成
[03/20] 2277-149896-0002 推理完成
[04/20] 2277-149896-0003 推理完成


2026-11-13 02:48:59,650 - INFO - Loaded d:\Projects\speech-ai-learning\data\w08_english_dev_20\006_2277-149896-0005.wav,shape=torch.Size([1, 89600]), sr=16000
2026-11-13 02:48:59,777 - INFO - Loaded d:\Projects\speech-ai-learning\data\w08_english_dev_20\007_2277-149896-0006.wav,shape=torch.Size([1, 79520]), sr=16000
2026-11-13 02:48:59,843 - INFO - Loaded d:\Projects\speech-ai-learning\data\w08_english_dev_20\008_2277-149896-0007.wav,shape=torch.Size([1, 86000]), sr=16000


[05/20] 2277-149896-0004 推理完成
[06/20] 2277-149896-0005 推理完成
[07/20] 2277-149896-0006 推理完成


2026-11-13 02:48:59,877 - INFO - Loaded d:\Projects\speech-ai-learning\data\w08_english_dev_20\009_2277-149896-0008.wav,shape=torch.Size([1, 111040]), sr=16000
2026-11-13 02:48:59,917 - INFO - Loaded d:\Projects\speech-ai-learning\data\w08_english_dev_20\010_2277-149896-0009.wav,shape=torch.Size([1, 93040]), sr=16000
2026-11-13 02:48:59,953 - INFO - Loaded d:\Projects\speech-ai-learning\data\w08_english_dev_20\011_2277-149896-0010.wav,shape=torch.Size([1, 84400]), sr=16000
2026-11-13 02:48:59,987 - INFO - Loaded d:\Projects\speech-ai-learning\data\w08_english_dev_20\012_2277-149896-0011.wav,shape=torch.Size([1, 71440]), sr=16000
2026-11-13 02:49:00,027 - INFO - Loaded d:\Projects\speech-ai-learning\data\w08_english_dev_20\013_2277-149896-0012.wav,shape=torch.Size([1, 39920]), sr=16000
2026-11-13 02:49:00,059 - INFO - Loaded d:\Projects\speech-ai-learning\data\w08_english_dev_20\014_2277-149896-0013.wav,shape=torch.Size([1, 97440]), sr=16000


[08/20] 2277-149896-0007 推理完成
[09/20] 2277-149896-0008 推理完成
[10/20] 2277-149896-0009 推理完成
[11/20] 2277-149896-0010 推理完成
[12/20] 2277-149896-0011 推理完成
[13/20] 2277-149896-0012 推理完成


2026-11-13 02:49:00,092 - INFO - Loaded d:\Projects\speech-ai-learning\data\w08_english_dev_20\015_2277-149896-0014.wav,shape=torch.Size([1, 65120]), sr=16000
2026-11-13 02:49:00,123 - INFO - Loaded d:\Projects\speech-ai-learning\data\w08_english_dev_20\016_2277-149896-0015.wav,shape=torch.Size([1, 58800]), sr=16000
2026-11-13 02:49:00,154 - INFO - Loaded d:\Projects\speech-ai-learning\data\w08_english_dev_20\017_2277-149896-0016.wav,shape=torch.Size([1, 44560]), sr=16000
2026-11-13 02:49:00,185 - INFO - Loaded d:\Projects\speech-ai-learning\data\w08_english_dev_20\018_2277-149896-0017.wav,shape=torch.Size([1, 79760]), sr=16000
2026-11-13 02:49:00,221 - INFO - Loaded d:\Projects\speech-ai-learning\data\w08_english_dev_20\019_2277-149896-0018.wav,shape=torch.Size([1, 72720]), sr=16000
2026-11-13 02:49:00,252 - INFO - Loaded d:\Projects\speech-ai-learning\data\w08_english_dev_20\020_2277-149896-0019.wav,shape=torch.Size([1, 56960]), sr=16000


[14/20] 2277-149896-0013 推理完成
[15/20] 2277-149896-0014 推理完成
[16/20] 2277-149896-0015 推理完成
[17/20] 2277-149896-0016 推理完成
[18/20] 2277-149896-0017 推理完成
[19/20] 2277-149896-0018 推理完成
[20/20] 2277-149896-0019 推理完成

全部推理完成！


In [18]:
#保存预测结果
prediction_df = pd.DataFrame(prediction_rows)

results_dir = PROJECT_ROOT / "results"
results_dir.mkdir(parents=True, exist_ok=True)

output_path = (
    results_dir
    / "w08_wav2vec2_english_predictions.csv"
)

prediction_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("保存完成！")
print("文件位置:", output_path)
print("保存数量:", len(prediction_df))

保存完成！
文件位置: D:\Projects\speech-ai-learning\results\w08_wav2vec2_english_predictions.csv
保存数量: 20


In [19]:
#结果检查与验收
print("===== Result Validation =====")

print("样本数量:", len(prediction_df))

print(
    "UID 是否唯一:",
    prediction_df["id"].is_unique
)

print(
    "Reference 空值数量:",
    prediction_df["reference"].isna().sum()
)

print(
    "Hypothesis 空值数量:",
    prediction_df["hypothesis"].isna().sum()
)

print(
    "空预测数量:",
    prediction_df["hypothesis"].fillna("").str.strip().eq("").sum()
)

print(
    "Checkpoint:",
    prediction_df["checkpoint"].unique()
)

print(
    "Device:",
    prediction_df["device"].unique()
)

print("\n===== Prediction Preview =====")

for _, row in prediction_df.head(5).iterrows():
    print("\nID:", row["id"])
    print("Reference:", row["reference"])
    print("Hypothesis:", row["hypothesis"])
    print("-" * 70)

===== Result Validation =====
样本数量: 20
UID 是否唯一: True
Reference 空值数量: 0
Hypothesis 空值数量: 0
空预测数量: 0
Checkpoint: <ArrowStringArray>
['facebook/wav2vec2-base-960h']
Length: 1, dtype: str
Device: <ArrowStringArray>
['cuda']
Length: 1, dtype: str

===== Prediction Preview =====

ID: 2277-149896-0000
Reference: HE WAS IN A FEVERED STATE OF MIND OWING TO THE BLIGHT HIS WIFE'S ACTION THREATENED TO CAST UPON HIS ENTIRE FUTURE
Hypothesis: HE WAS IN A FEVERED STATE OF MIND OWING TO THE BLIGHT HIS WIFE'S ACTION THREATENED TO CAST UPON HIS ENTIRE FUTURE
----------------------------------------------------------------------

ID: 2277-149896-0001
Reference: HE WOULD HAVE TO PAY HER THE MONEY WHICH SHE WOULD NOW REGULARLY DEMAND OR THERE WOULD BE TROUBLE IT DID NOT MATTER WHAT HE DID
Hypothesis: HE WOULD HAVE TO PAY HER THE MONEY WHICH SHE WOULD NOW REGULARLY DEMAND OR THERE WOULD BE TROUBLE IT DID NOT MATTER WHAT HE DID
----------------------------------------------------------------------

ID: 2277

### whisper推理

In [21]:
#释放ctc模型
import gc
import torch

# 释放 wav2vec2 模型
if "model" in globals():
    del model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("GPU cache cleared.")

GPU cache cleared.


In [22]:
#导入W7的Whisper接口
from src.asr_inference import (
    load_whisper_model,
    transcribe_file,
)

import inspect

print("load_whisper_model:")
print(inspect.signature(load_whisper_model))

print("\ntranscribe_file:")
print(inspect.signature(transcribe_file))

load_whisper_model:
(checkpoint='openai/whisper-small', device=None)

transcribe_file:
(audio_path, processor, model, device=None, language='zh', task='transcribe', max_new_tokens=96)


In [23]:
#加载whisper-small
WHISPER_CHECKPOINT = "openai/whisper-small"

whisper_processor, whisper_model, whisper_device = (
    load_whisper_model(
        checkpoint=WHISPER_CHECKPOINT
    )
)

print("Checkpoint:", WHISPER_CHECKPOINT)
print("Device:", whisper_device)

print(
    "Sampling rate:",
    whisper_processor.feature_extractor.sampling_rate
)

2026-11-13 02:59:42,499 - INFO - HTTP Request: GET https://huggingface.co/api/models/openai/whisper-small "HTTP/1.1 200 OK"
2026-11-13 02:59:42,845 - INFO - HTTP Request: GET https://huggingface.co/api/models/openai/whisper-small/tree/973afd24965f72e36ca33b3055d56a652f456b4d/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-11-13 02:59:43,636 - INFO - HTTP Request: GET https://huggingface.co/api/models/openai/whisper-small/tree/973afd24965f72e36ca33b3055d56a652f456b4d/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-11-13 02:59:43,982 - INFO - HTTP Request: GET https://huggingface.co/api/models/openai/whisper-small/tree/973afd24965f72e36ca33b3055d56a652f456b4d?recursive=true&expand=false "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Checkpoint: openai/whisper-small
Device: cuda
Checkpoint: openai/whisper-small
Device: cuda
Sampling rate: 16000


In [24]:
#先推理第一条音频
#读取第一条音频
metadata_path = (
    PROJECT_ROOT
    / "metadata"
    / "w08_english_dev_20.csv"
)

english_df = pd.read_csv(metadata_path)

assert len(english_df) == 20

first_row = english_df.iloc[0]

first_audio_path = Path(first_row["audio_path"])

if not first_audio_path.is_absolute():
    first_audio_path = PROJECT_ROOT / first_audio_path

print("Audio:", first_audio_path)
print("Reference:", first_row["reference"])

Audio: d:\Projects\speech-ai-learning\data\w08_english_dev_20\001_2277-149896-0000.wav
Reference: HE WAS IN A FEVERED STATE OF MIND OWING TO THE BLIGHT HIS WIFE'S ACTION THREATENED TO CAST UPON HIS ENTIRE FUTURE


In [26]:
#调用WHISPER推理接口
whisper_result = transcribe_file(
    audio_path=first_audio_path,
    processor=whisper_processor,
    model=whisper_model,
    device=whisper_device,
    language="en",
    task="transcribe",
    max_new_tokens=96
)

print("\n========== Whisper Result ==========")

print("\nReference:")
print(first_row["reference"])

print("\nHypothesis:")
print(whisper_result["transcript"])

print("\nGenerated IDs:")
print(whisper_result["generated_ids"][:30])

print("\nMetadata:")

for key, value in whisper_result["metadata"].items():
    print(f"{key}: {value}")

2026-11-13 03:02:04,278 - INFO - Loaded d:\Projects\speech-ai-learning\data\w08_english_dev_20\001_2277-149896-0000.wav,shape=torch.Size([1, 105440]), sr=16000
[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=96) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



========== Whisper Result ==========

Reference:
HE WAS IN A FEVERED STATE OF MIND OWING TO THE BLIGHT HIS WIFE'S ACTION THREATENED TO CAST UPON HIS ENTIRE FUTURE

Hypothesis:
 He was in a fevered state of mind, owing to the blight his wife's action threatened to cast upon his entire future.

Generated IDs:
[634, 390, 294, 257, 18277, 292, 1785, 295, 1575, 11, 277, 7904, 281, 264, 888, 397, 702, 3836, 311, 3069, 18268, 281, 4193, 3564, 702, 2302, 2027, 13]

Metadata:
audio_path: d:\Projects\speech-ai-learning\data\w08_english_dev_20\001_2277-149896-0000.wav
checkpoint: openai/whisper-small
original_sampling_rate: 16000
model_sampling_rate: 16000
duration_sec: 6.59
device: cuda
input_features_shape: (1, 80, 3000)
generated_ids_shape: (1, 28)


In [ ]:
#20条英文音频批量推理
from pathlib import Path
import pandas as pd

whisper_prediction_rows = []

for index, row in english_df.iterrows():
    # 1. 获取音频路径
    audio_path = Path(row["audio_path"])

    if not audio_path.is_absolute():
        audio_path = PROJECT_ROOT / audio_path

    if not audio_path.exists():
        raise FileNotFoundError(audio_path)

    # 2. 调用已有 Whisper 推理接口
    result = transcribe_file(
        audio_path=audio_path,
        processor=whisper_processor,
        model=whisper_model,
        device=whisper_device,
        language="en",
        task="transcribe",
        max_new_tokens=96,
    )

    #3.获取预测文本和元数据
    hypothesis = result["transcript"]
    meta = result["metadata"]

    # 4. 保存结果
    whisper_prediction_rows.append({
        "id": row["id"],
        "speaker_id": row["speaker_id"],
        "reference": row["reference"],
        "hypothesis": hypothesis,
        "audio_path": row["audio_path"],
        "checkpoint": WHISPER_CHECKPOINT,
        "device": str(whisper_device),
        "language": "en",
        "task": "transcribe",
        "duration_sec": meta.get("duration_sec"),
        "generated_ids_length": len(
            result["generated_ids"]
        ),
    })

    print(
        f"[{index + 1:02d}/20] "
        f"{row['id']} 推理完成"
    )

print("\nWhisper 全部推理完成！")

In [28]:
#保存whisper预测结果
whisper_prediction_df = pd.DataFrame(
    whisper_prediction_rows
)

results_dir = PROJECT_ROOT / "results"
results_dir.mkdir(parents=True, exist_ok=True)

whisper_output_path = (
    results_dir
    / "w08_whisper_english_predictions.csv"
)

whisper_prediction_df.to_csv(
    whisper_output_path,
    index=False,
    encoding="utf-8-sig",
)

print("保存成功！")
print("文件路径:", whisper_output_path)
print("样本数量:", len(whisper_prediction_df))

保存成功！
文件路径: D:\Projects\speech-ai-learning\results\w08_whisper_english_predictions.csv
样本数量: 20


In [29]:
#结果完整性检查
print("===== Whisper Result Validation =====")

print("样本数量:", len(whisper_prediction_df))

print(
    "UID 是否唯一:",
    whisper_prediction_df["id"].is_unique
)

print(
    "Reference 空值数量:",
    whisper_prediction_df["reference"].isna().sum()
)

print(
    "Hypothesis 空值数量:",
    whisper_prediction_df["hypothesis"].isna().sum()
)

print(
    "空预测数量:",
    whisper_prediction_df["hypothesis"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print(
    "Checkpoint:",
    whisper_prediction_df["checkpoint"].unique()
)

print(
    "Language:",
    whisper_prediction_df["language"].unique()
)

print("\n===== Prediction Preview =====")

for _, row in whisper_prediction_df.head(5).iterrows():

    print("\nID:", row["id"])
    print("Reference:", row["reference"])
    print("Hypothesis:", row["hypothesis"])
    print("-" * 70)

===== Whisper Result Validation =====
样本数量: 20
UID 是否唯一: True
Reference 空值数量: 0
Hypothesis 空值数量: 0
空预测数量: 0
Checkpoint: <ArrowStringArray>
['openai/whisper-small']
Length: 1, dtype: str
Language: <ArrowStringArray>
['en']
Length: 1, dtype: str

===== Prediction Preview =====

ID: 2277-149896-0000
Reference: HE WAS IN A FEVERED STATE OF MIND OWING TO THE BLIGHT HIS WIFE'S ACTION THREATENED TO CAST UPON HIS ENTIRE FUTURE
Hypothesis:  He was in a fevered state of mind, owing to the blight his wife's action threatened to cast upon his entire future.
----------------------------------------------------------------------

ID: 2277-149896-0001
Reference: HE WOULD HAVE TO PAY HER THE MONEY WHICH SHE WOULD NOW REGULARLY DEMAND OR THERE WOULD BE TROUBLE IT DID NOT MATTER WHAT HE DID
Hypothesis:  He would have to pay her the money which she would now regularly demand or there would be trouble. It did not matter what he did.
----------------------------------------------------------------------

I

In [30]:
#确认两个模型使用的是同一批 UID 和 Reference
ctc_df = pd.read_csv(
    PROJECT_ROOT
    / "results"
    / "w08_wav2vec2_english_predictions.csv"
)

whisper_df = pd.read_csv(whisper_output_path)

# 检查 UID 集合
assert set(ctc_df["id"]) == set(whisper_df["id"])

# 按 UID 对齐后检查 Reference
merged_check = ctc_df[
    ["id", "reference"]
].merge(
    whisper_df[["id", "reference"]],
    on="id",
    suffixes=("_ctc", "_whisper"),
    validate="one_to_one",
)

assert (
    merged_check["reference_ctc"]
    == merged_check["reference_whisper"]
).all()

print("两个模型使用相同的20条音频和Reference！")

两个模型使用相同的20条音频和Reference！


### 5.比较二者WER

In [31]:
#合并两个模型的结果
from pathlib import Path
import pandas as pd

ROOT = Path(r"D:\Projects\speech-ai-learning")

source_df = pd.read_csv(
    ROOT / "metadata/w08_english_dev_20.csv",
    dtype={"id": str},
)

ctc_df = pd.read_csv(
    ROOT / "results/w08_wav2vec2_english_predictions.csv",
    dtype={"id": str},
)

whisper_df = pd.read_csv(
    ROOT / "results/w08_whisper_english_predictions.csv",
    dtype={"id": str},
)

# 样本数量及 UID 检查
for name, frame in [
    ("Source", source_df),
    ("CTC", ctc_df),
    ("Whisper", whisper_df),
]:
    assert len(frame) == 20, f"{name} 样本数量错误"
    assert frame["id"].is_unique, f"{name} 存在重复 UID"

assert set(source_df["id"]) == set(ctc_df["id"])
assert set(source_df["id"]) == set(whisper_df["id"])

# 将预测结果列重命名，避免混淆
ctc_part = ctc_df[
    ["id", "reference", "hypothesis", "audio_path"]
].rename(columns={
    "reference": "ctc_reference",
    "hypothesis": "ctc_hypothesis",
    "audio_path": "ctc_audio_path",
})

whisper_part = whisper_df[
    ["id", "reference", "hypothesis", "audio_path"]
].rename(columns={
    "reference": "whisper_reference",
    "hypothesis": "whisper_hypothesis",
    "audio_path": "whisper_audio_path",
})

# 根据 UID 合并
comparison_df = (
    source_df[
        ["id", "speaker_id", "audio_path", "reference"]
    ]
    .merge(ctc_part, on="id", validate="one_to_one")
    .merge(whisper_part, on="id", validate="one_to_one")
)

# Reference 必须完全相同
assert (
    comparison_df["reference"]
    == comparison_df["ctc_reference"]
).all()

assert (
    comparison_df["reference"]
    == comparison_df["whisper_reference"]
).all()

# 音频路径必须对应同一个样本
assert (
    comparison_df["audio_path"]
    == comparison_df["ctc_audio_path"]
).all()

assert (
    comparison_df["audio_path"]
    == comparison_df["whisper_audio_path"]
).all()

print("合并后的样本数量:", len(comparison_df))
print("UID、Reference、Audio Path 一致性检查通过！")

合并后的样本数量: 20
UID、Reference、Audio Path 一致性检查通过！


In [32]:
#统一规范化并计算
from src.normalize import normalize_score
from src.metrics import score_tokens, score_corpus

# 统一评分文本
comparison_df["score_reference"] = (
    comparison_df["reference"].map(normalize_score)
)

comparison_df["score_ctc_hypothesis"] = (
    comparison_df["ctc_hypothesis"].map(normalize_score)
)

comparison_df["score_whisper_hypothesis"] = (
    comparison_df["whisper_hypothesis"].map(normalize_score)
)

# 统一使用英文空格分词
reference_tokens = [
    text.split()
    for text in comparison_df["score_reference"]
]

ctc_tokens = [
    text.split()
    for text in comparison_df["score_ctc_hypothesis"]
]

whisper_tokens = [
    text.split()
    for text in comparison_df["score_whisper_hypothesis"]
]

# 生成语料级评分输入
ctc_pairs = list(zip(reference_tokens, ctc_tokens))
whisper_pairs = list(zip(reference_tokens, whisper_tokens))

# 使用已有评分函数
ctc_result = score_corpus(ctc_pairs)
whisper_result = score_corpus(whisper_pairs)

# 检查两个模型参考词总数相同
assert ctc_result["N"] == whisper_result["N"]
assert ctc_result["N"] > 0

# 记录逐条评分
for name, pairs in [
    ("ctc", ctc_pairs),
    ("whisper", whisper_pairs),
]:
    individual_results = [
        score_tokens(ref, hyp)
        for ref, hyp in pairs
    ]

    for key in ["S", "D", "I", "N", "distance"]:
        comparison_df[f"{name}_{key}"] = [
            result[key] for result in individual_results
        ]

# 核对逐条累计值与语料级评分是否一致
for name, result in [
    ("ctc", ctc_result),
    ("whisper", whisper_result),
]:
    for key in ["S", "D", "I", "N", "distance"]:
        assert comparison_df[f"{name}_{key}"].sum() == result[key]

print("===== Corpus-level WER =====")

for name, result in [
    ("wav2vec2-base-960h", ctc_result),
    ("whisper-small", whisper_result),
]:
    print(f"\nModel: {name}")
    print("S:", result["S"])
    print("D:", result["D"])
    print("I:", result["I"])
    print("N:", result["N"])
    print("WER:", f"{result['error_rate']:.2%}")

===== Corpus-level WER =====

Model: wav2vec2-base-960h
S: 6
D: 0
I: 0
N: 321
WER: 1.87%

Model: whisper-small
S: 9
D: 3
I: 0
N: 321
WER: 3.74%


In [33]:
#分析识别错误
# 只显示至少一个模型有错误的样本
error_df = comparison_df[
    (comparison_df["ctc_distance"] > 0)
    | (comparison_df["whisper_distance"] > 0)
]

print("出现识别错误的样本数量:", len(error_df))

for _, row in error_df.iterrows():

    print("\n" + "=" * 65)
    print("UID:", row["id"])

    print("\nReference:")
    print(row["reference"])

    print("\nCTC Hypothesis:")
    print(row["ctc_hypothesis"])
    print("CTC Distance:", row["ctc_distance"])

    print("\nWhisper Hypothesis:")
    print(row["whisper_hypothesis"])
    print("Whisper Distance:", row["whisper_distance"])

出现识别错误的样本数量: 8

UID: 2277-149896-0006

Reference:
HE COULD ARRANGE THAT SATISFACTORILY FOR CARRIE WOULD BE GLAD TO WAIT IF NECESSARY

CTC Hypothesis:
HE COULD ARRANGE THAT SATISFACTORIY FOR CARREY WOULD BE GLAD TO WAIT IF NECESSARY
CTC Distance: 2

Whisper Hypothesis:
 He could arrange that satisfactorily, for Kerry would be glad to wait if necessary.
Whisper Distance: 1

UID: 2277-149896-0007

Reference:
HE WOULD SEE HOW THINGS TURNED OUT TO MORROW AND THEN HE WOULD TALK TO HER THEY WERE GOING TO MEET AS USUAL

CTC Hypothesis:
HE WOULD SEE HOW THINGS TURNED OUT TO MORROW AND THEN HE WOULD TALK TO HER THEY WERE GOING TO MEET AS USUAL
CTC Distance: 0

Whisper Hypothesis:
 He would see how things turned out tomorrow, and then he would talk to her. They were going to meet as usual.
Whisper Distance: 2

UID: 2277-149896-0009

Reference:
WHILE THE DANGER HAD NOT LESSENED IT HAD NOT AS YET MATERIALISED AND WITH HIM NO NEWS WAS GOOD NEWS

CTC Hypothesis:
WHILE THE DANGER HAD NOT LESSENED IT H

In [34]:
#保存最终 CSV 与实验元数据
import json
import hashlib
import src.normalize as normalize_module
import src.metrics as metrics_module

results_dir = ROOT / "results"
results_dir.mkdir(parents=True, exist_ok=True)

output_path = results_dir / "w08_english_ctc.csv"
metadata_path = results_dir / "w08_english_ctc_metadata.json"

# 模型信息
comparison_df["ctc_checkpoint"] = (
    "facebook/wav2vec2-base-960h"
)
comparison_df["whisper_checkpoint"] = (
    "openai/whisper-small"
)

# 保存逐条结果
comparison_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig",
)

# 记录评分代码的实际内容哈希
def file_sha256(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()

metadata = {
    "dataset": "openslr/librispeech_asr",
    "configuration": "clean",
    "split": "validation",
    "sample_count": 20,
    "selection": "first_20",
    "speaker_count": int(
        comparison_df["speaker_id"].nunique()
    ),
    "chapter_count": 1,
    "ctc_checkpoint": "facebook/wav2vec2-base-960h",
    "whisper_checkpoint": "openai/whisper-small",
    "whisper_language": "en",
    "whisper_task": "transcribe",
    "normalizer_version": getattr(
        normalize_module, "NORMALIZER_VERSION", "unspecified"
    ),
    "normalizer_sha256": file_sha256(normalize_module.__file__),
    "scoring_sha256": file_sha256(metrics_module.__file__),
    "tokenization": "normalize_score_then_whitespace_split",
    "aggregation": "corpus_level",
    "ctc_metrics": {
        key: ctc_result[key]
        for key in ["S", "D", "I", "N", "distance", "error_rate"]
    },
    "whisper_metrics": {
        key: whisper_result[key]
        for key in ["S", "D", "I", "N", "distance", "error_rate"]
    },
    "limitations": [
        "All 20 clips come from speaker 2277, chapter 149896",
        "Small, non-random, correlated evaluation sample",
        "Model differences do not establish architecture causality",
        "English checkpoint excluded from Chinese mixed-language ranking",
    ],
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

# 重新读取检查
saved_df = pd.read_csv(output_path)

assert len(saved_df) == 20
assert saved_df["id"].is_unique
assert saved_df["reference"].notna().all()
assert saved_df["ctc_hypothesis"].notna().all()
assert saved_df["whisper_hypothesis"].notna().all()

print("最终结果保存成功！")
print("CSV:", output_path)
print("Metadata:", metadata_path)
print("样本数:", len(saved_df))

最终结果保存成功！
CSV: D:\Projects\speech-ai-learning\results\w08_english_ctc.csv
Metadata: D:\Projects\speech-ai-learning\results\w08_english_ctc_metadata.json
样本数: 20
